# Lag Selection — Wide Sweep

Exercise the deterministic autoregressive lag-selection pipeline across many
data shapes and frequencies to **hunt for strange lag selections**.

Pipeline under test (`skforecast_ai.recommendation.autoregressive`):

```
create_data_profile  ->  compute_series_pacf  ->  finalize_lags
                                                   (+ select_window_features)
```

We cover:

- **Single series**: minutely, hourly, daily, weekly, monthly, quarterly, yearly.
- **Multi-series** (wide, global model, `task_type="multi_series"`).
- **Multivariate** (wide, `task_type="multivariate"`).
- **Long format** (`series_id_column`).

Each series is synthetic with a **known** AR + seasonal structure, so the
selected lags can be compared against what we expect. A diagnostics table at the
end flags suspicious selections (missing primary season, excessive max lag,
runaway lag counts, etc.).

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
import warnings

import numpy as np
import pandas as pd

from skforecast_ai.profiling.data_profile import (
    create_data_profile,
    estimate_seasonality,
)
from skforecast_ai.recommendation.autoregressive import (
    compute_series_pacf,
    finalize_lags,
    select_window_features,
)

warnings.filterwarnings("ignore")
rng_global = np.random.default_rng(123)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

/opt/homebrew/Caskroom/miniconda/base/envs/skforecast_ai_py13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Synthetic series generator

`make_seasonal_series` builds a series with an AR(1) backbone plus one or two
sinusoidal seasonal components, so we know which lags *should* be picked up
(short AR lags + seasonal period lags).

In [3]:
def make_seasonal_series(
    n: int,
    seasonalities: list[int],
    ar_coef: float = 0.5,
    noise: float = 0.3,
    seed: int = 0,
) -> np.ndarray:
    """
    Generate an AR(1) + multi-seasonal synthetic series.

    Parameters
    ----------
    n : int
        Number of observations.
    seasonalities : list of int
        Seasonal periods to inject as sine waves (e.g. [24, 168]).
    ar_coef : float
        AR(1) coefficient on the stochastic backbone.
    noise : float
        Std of the innovation noise.
    seed : int
        RNG seed.

    Returns
    -------
    y : numpy ndarray
    """
    rng = np.random.default_rng(seed)
    t = np.arange(n)

    seasonal = np.zeros(n, dtype=float)
    for i, period in enumerate(seasonalities):
        amp = 1.0 / (i + 1)  # primary stronger than secondary
        seasonal += amp * np.sin(2 * np.pi * t / period)

    ar = np.zeros(n, dtype=float)
    for i in range(1, n):
        ar[i] = ar_coef * ar[i - 1] + rng.normal(0, noise)

    return seasonal + ar

In [4]:
def run_single(
    name: str,
    freq: str,
    n: int,
    seasonalities: list[int] | None = None,
    seed: int = 0,
) -> dict:
    """
    Build a single-series profile and run the full lag-selection pipeline.

    Returns a dict row summarising inputs, the seasonality estimate, the
    raw PACF-significant lags, and the finalized lags.
    """
    if seasonalities is None:
        seasonalities = estimate_seasonality(freq) or [max(2, n // 10)]

    y = make_seasonal_series(n, seasonalities, seed=seed)
    idx = pd.date_range("2015-01-01", periods=n, freq=freq)
    df = pd.DataFrame({"y": y}, index=idx)

    profile = create_data_profile(df, target="y")
    series_pacf = compute_series_pacf(df, profile)
    wf = select_window_features(profile.n_observations, profile.frequency, profile.target_stats)
    lags = finalize_lags(
        series_pacf     = series_pacf,
        task_type       = "single_series",
        n_observations  = profile.n_observations,
        frequency       = profile.frequency,
        window_features = wf,
    )

    pacf_lags = series_pacf[0].lags if series_pacf else []
    return {
        "case": name,
        "task": "single_series",
        "freq_in": freq,
        "freq_detected": profile.frequency,
        "n": n,
        "seasonalities": estimate_seasonality(profile.frequency),
        "n_pacf_lags": len(pacf_lags),
        "pacf_top10": sorted(pacf_lags)[:10],
        "n_final_lags": len(lags),
        "max_lag": max(lags) if lags else None,
        "final_lags": lags,
        "window_features": wf,
    }

## 2. Single series across frequencies

Generous lengths so the `n_lags_cap` ceiling and `n // 3` constraint both come
into play depending on frequency.

In [5]:
single_cases = [
    # name,           freq,    n
    ("minutely_3w",   "min",   3 * 7 * 24 * 60 // 4),   # ~7560 obs
    ("hourly_3mo",    "h",     24 * 90),                # 2160 obs
    ("daily_4y",      "D",     365 * 4),                # 1460 obs
    ("daily_2y",      "D",     365 * 2),                # 730 obs (stresses lag 365)
    ("weekly_5y",     "W",     52 * 5),                 # 260 obs
    ("monthly_10y",   "MS",    12 * 10),                # 120 obs
    ("quarterly_15y", "QS",    4 * 15),                 # 60 obs
    ("yearly_50y",    "YS",    50),                     # 50 obs
]

rows = [run_single(name, freq, n, seed=i) for i, (name, freq, n) in enumerate(single_cases)]
single_df = pd.DataFrame(rows)
single_df[
    ["case", "freq_detected", "n", "seasonalities", "n_pacf_lags", "n_final_lags", "max_lag", "final_lags"]
]

,case,freq_detected,n,seasonalities,n_pacf_lags,n_final_lags,max_lag,final_lags
0,minutely_3w,min,7560,"[60, 1440]",46,47,60,"[1, 2, 3, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16,..."
1,hourly_3mo,h,2160,"[24, 168]",16,17,30,"[1, 3, 4, 5, 6, 7, 8, 13, 14, 15, 16, 17, 18, ..."
2,daily_4y,D,1460,"[7, 365]",10,6,20,"[1, 3, 7, 12, 13, 20]"
3,daily_2y,D,730,"[7, 365]",8,4,20,"[1, 7, 13, 20]"
4,weekly_5y,W-SUN,260,[52],1,6,52,"[1, 2, 3, 4, 5, 52]"
5,monthly_10y,MS,120,[12],4,5,12,"[1, 2, 3, 4, 12]"
6,quarterly_15y,QS-OCT,60,[4],2,4,5,"[1, 2, 4, 5]"
7,yearly_50y,YS-JAN,50,[1],1,5,5,"[1, 2, 3, 4, 5]"


In [17]:
from skforecast.stats import calculate_lag_autocorrelation

from scipy.stats import norm
from skforecast_ai.profiling.data_profile import estimate_seasonality
from skforecast_ai.schemas import DataProfile, SeriesPacf

# PACF horizon (`n_lags_cap`) bounds. The cap scales with the largest seasonal
# period (`3 * seasonalities[-1]`) so PACF can see a few seasonal cycles, but is
# clamped to `MAX_PACF_CAP` to bound compute on fine-grained frequencies (e.g.
# minutely, whose 3-day reach would otherwise be 4320 lags). Seasonal lags
# beyond the ceiling are not lost: `select_lags` re-adds them via seasonal
# enrichment. `DEFAULT_PACF_CAP` is the fallback when the frequency is unknown
# or carries no seasonality.
DEFAULT_PACF_CAP = 50
MAX_PACF_CAP = 512


def compute_series_pacf_2(
    data: pd.DataFrame,
    profile,
) -> list:
    """
    """

    seasonalities = estimate_seasonality(profile.frequency)
    n_lags_cap = (
        min(3 * seasonalities[-1], MAX_PACF_CAP)
        if seasonalities else DEFAULT_PACF_CAP
    )

    if isinstance(profile.target, list):
        # Wide format: each target column is a series.
        raw_series = [(col, data[col]) for col in profile.target]
    elif profile.series_id_column is not None:
        # Long format: group by series id.
        raw_series = [
            (str(sid), group[profile.target])
            for sid, group in data.groupby(profile.series_id_column)
        ]
    else:
        # Single series.
        raw_series = [(profile.target, data[profile.target])]

    results: list[SeriesPacf] = []
    for series_id, raw in raw_series:
        n = int(raw.count())
        if n < 4:
            continue

        # PACF requires n_lags < n_finite // 2.
        n_lags = max(min(n_lags_cap, n // 2 - 1), 1)
        lag_table = calculate_lag_autocorrelation(
            data=raw, n_lags=n_lags, sort_by="partial_autocorrelation_abs"
        )
        # Bonferroni-corrected white-noise threshold: split the 5 %
        # family-wise error across the `n_lags` tested lags so isolated
        # far-lag noise no longer passes as significant.
        alpha = 0.05 / n_lags
        threshold = norm.ppf(1 - alpha / 2) / np.sqrt(n)
        significant = lag_table.loc[
            lag_table["partial_autocorrelation_abs"] > threshold
        ]

        results.append(lag_table)

    print(n_lags, n_lags_cap, threshold)

    return results

In [18]:
def run_single_2(
    name: str,
    freq: str,
    n: int,
    seasonalities: list[int] | None = None,
    seed: int = 0,
) -> dict:
    """
    Build a single-series profile and run the full lag-selection pipeline.

    Returns a dict row summarising inputs, the seasonality estimate, the
    raw PACF-significant lags, and the finalized lags.
    """
    if seasonalities is None:
        seasonalities = estimate_seasonality(freq) or [max(2, n // 10)]

    y = make_seasonal_series(n, seasonalities, seed=seed)
    idx = pd.date_range("2015-01-01", periods=n, freq=freq)
    df = pd.DataFrame({"y": y}, index=idx)

    profile = create_data_profile(df, target="y")
    series_pacf = compute_series_pacf_2(df, profile)

    return series_pacf

In [19]:
single_cases = [
    # name,           freq,    n
    ("minutely_3w",   "min",   3 * 7 * 24 * 60 // 4),   # ~7560 obs
    ("hourly_3mo",    "h",     24 * 90),                # 2160 obs
    ("daily_4y",      "D",     365 * 4),                # 1460 obs
    ("daily_2y",      "D",     365 * 2),                # 730 obs (stresses lag 365)
    ("weekly_5y",     "W",     52 * 5),                 # 260 obs
    ("monthly_10y",   "MS",    12 * 10),                # 120 obs
    ("quarterly_15y", "QS",    4 * 15),                 # 60 obs
    ("yearly_50y",    "YS",    50),                     # 50 obs
]

rows = [run_single_2(name, freq, n, seed=i) for i, (name, freq, n) in enumerate(single_cases)]
single_df = pd.DataFrame(rows[2][0])
single_df

512 512 0.04481219648589872
504 504 0.08375379875769025
512 512 0.10197194460865272
364 512 0.14112063233791322
129 156 0.2200616776699584
36 36 0.29184029261941624
12 12 0.36990350621284157
3 3 0.33855987009505645


,lag,partial_autocorrelation_abs,partial_autocorrelation,autocorrelation_abs,autocorrelation
0,2,0.682560,-0.682560,0.055845,0.055845
1,1,0.662464,0.662464,0.662464,0.662464
2,5,0.461192,0.461192,0.009552,0.009552
3,6,0.352571,0.352571,0.570424,0.570424
4,4,0.348954,0.348954,0.436115,-0.436115
...,...,...,...,...,...
507,474,0.000231,-0.000231,0.134405,-0.134405
508,165,0.000142,0.000142,0.678128,-0.678128
509,488,0.000095,-0.000095,0.162572,-0.162572
510,286,0.000083,0.000083,0.348730,0.348730


## 3. Multi-series (wide, global model)

Several related series sharing structure. `task_type="multi_series"` triggers the
**consensus** aggregation (a lag must be significant in at least half the series).

In [6]:
def run_wide(
    name: str,
    freq: str,
    n: int,
    n_series: int,
    task_type: str,
    seasonalities: list[int] | None = None,
) -> dict:
    """Build a wide multi-series profile and run lag selection for a task_type."""
    if seasonalities is None:
        seasonalities = estimate_seasonality(freq) or [max(2, n // 10)]

    idx = pd.date_range("2015-01-01", periods=n, freq=freq)
    data = {
        f"s{j}": make_seasonal_series(n, seasonalities, seed=100 + j, noise=0.3 + 0.05 * j)
        for j in range(n_series)
    }
    df = pd.DataFrame(data, index=idx)
    targets = list(df.columns)

    profile = create_data_profile(df, target=targets)
    series_pacf = compute_series_pacf(df, profile)
    wf = select_window_features(profile.n_observations, profile.frequency, profile.target_stats)
    lags = finalize_lags(
        series_pacf     = series_pacf,
        task_type       = task_type,
        n_observations  = profile.n_observations,
        frequency       = profile.frequency,
        window_features = wf,
    )
    return {
        "case": name,
        "task": task_type,
        "freq_detected": profile.frequency,
        "n": n,
        "n_series": n_series,
        "seasonalities": estimate_seasonality(profile.frequency),
        "n_final_lags": len(lags),
        "max_lag": max(lags) if lags else None,
        "final_lags": lags,
    }

In [7]:
multi_cases = [
    ("hourly_multi",  "h",  24 * 60, 5),
    ("daily_multi",   "D",  365 * 3, 4),
    ("weekly_multi",  "W",  52 * 4,  6),
    ("monthly_multi", "MS", 12 * 8,  3),
]

multi_rows = [run_wide(name, freq, n, k, "multi_series") for name, freq, n, k in multi_cases]
multi_df = pd.DataFrame(multi_rows)
multi_df[["case", "freq_detected", "n", "n_series", "seasonalities", "n_final_lags", "max_lag", "final_lags"]]

,case,freq_detected,n,n_series,seasonalities,n_final_lags,max_lag,final_lags
0,hourly_multi,h,1440,5,"[24, 168]",12,24,"[1, 4, 5, 6, 7, 8, 15, 16, 17, 18, 19, 24]"
1,daily_multi,D,1095,4,"[7, 365]",8,20,"[1, 3, 7, 9, 12, 13, 19, 20]"
2,weekly_multi,W-SUN,208,6,[52],6,52,"[1, 2, 3, 4, 5, 52]"
3,monthly_multi,MS,96,3,[12],6,12,"[1, 2, 3, 4, 5, 12]"


## 4. Multivariate (wide, top-n union)

Same wide data, but `task_type="multivariate"` triggers the **top-n union**
aggregation (each series contributes its strongest lags). This usually yields a
broader lag set than consensus — a good place to spot runaway selections.

In [8]:
mv_cases = [
    ("hourly_mv",  "h",  24 * 60, 4),
    ("daily_mv",   "D",  365 * 3, 3),
    ("weekly_mv",  "W",  52 * 4,  5),
    ("monthly_mv", "MS", 12 * 8,  3),
]

mv_rows = [run_wide(name, freq, n, k, "multivariate") for name, freq, n, k in mv_cases]
mv_df = pd.DataFrame(mv_rows)
mv_df[["case", "freq_detected", "n", "n_series", "seasonalities", "n_final_lags", "max_lag", "final_lags"]]

,case,freq_detected,n,n_series,seasonalities,n_final_lags,max_lag,final_lags
0,hourly_mv,h,1440,4,"[24, 168]",19,54,"[1, 3, 4, 5, 6, 7, 8, 9, 13, 14, 15, 16, 17, 1..."
1,daily_mv,D,1095,3,"[7, 365]",8,19,"[1, 3, 7, 9, 11, 12, 13, 19]"
2,weekly_mv,W-SUN,208,5,[52],4,52,"[1, 2, 11, 52]"
3,monthly_mv,MS,96,3,[12],4,12,"[1, 2, 3, 12]"


## 5. Long format (`series_id_column`)

Long layout with an explicit id column. `n_observations` is the **shortest**
series, which gates the `n // 3` max-lag constraint — useful for spotting cases
where uneven series lengths shrink the lag set unexpectedly.

In [9]:
def run_long(name: str, freq: str, lengths: dict[str, int], seasonalities: list[int] | None = None) -> dict:
    """Build a long-format profile (ragged series) and run consensus lag selection."""
    if seasonalities is None:
        seasonalities = estimate_seasonality(freq) or [7]

    frames = []
    for j, (sid, n) in enumerate(lengths.items()):
        idx = pd.date_range("2015-01-01", periods=n, freq=freq)
        y = make_seasonal_series(n, seasonalities, seed=200 + j)
        frames.append(pd.DataFrame({"date": idx, "series_id": sid, "y": y}))
    df = pd.concat(frames, ignore_index=True)

    profile = create_data_profile(
        df, target="y", date_column="date", series_id_column="series_id"
    )
    series_pacf = compute_series_pacf(df, profile)
    wf = select_window_features(profile.n_observations, profile.frequency, profile.target_stats)
    lags = finalize_lags(
        series_pacf     = series_pacf,
        task_type       = "multi_series",
        n_observations  = profile.n_observations,
        frequency       = profile.frequency,
        window_features = wf,
    )
    return {
        "case": name,
        "task": "multi_series (long)",
        "freq_detected": profile.frequency,
        "n_shortest": profile.n_observations,
        "lengths": lengths,
        "seasonalities": estimate_seasonality(profile.frequency),
        "n_final_lags": len(lags),
        "max_lag": max(lags) if lags else None,
        "final_lags": lags,
    }

In [10]:
long_rows = [
    run_long("daily_ragged", "D", {"a": 365 * 3, "b": 365 * 2, "c": 200}),
    run_long("hourly_ragged", "h", {"a": 24 * 60, "b": 24 * 30, "c": 24 * 10}),
    run_long("weekly_even", "W", {"a": 52 * 4, "b": 52 * 4, "c": 52 * 4}),
]
long_df = pd.DataFrame(long_rows)
long_df[["case", "freq_detected", "n_shortest", "lengths", "seasonalities", "n_final_lags", "max_lag", "final_lags"]]

,case,freq_detected,n_shortest,lengths,seasonalities,n_final_lags,max_lag,final_lags
0,daily_ragged,D,200,"{'a': 1095, 'b': 730, 'c': 200}","[7, 365]",3,13,"[1, 7, 13]"
1,hourly_ragged,h,240,"{'a': 1440, 'b': 720, 'c': 240}","[24, 168]",8,24,"[1, 3, 4, 5, 15, 18, 19, 24]"
2,weekly_even,W-SUN,208,"{'a': 208, 'b': 208, 'c': 208}",[52],6,52,"[1, 2, 3, 4, 5, 52]"


## 6. Diagnostics — flag strange selections

Heuristic checks applied to every case. Any `True` flag is worth a manual look.
Two of the checks are **seasonality- and scale-aware** so they stop firing on
selections that are large *by design*:

- **`missing_primary`**: the primary seasonal lag is absent from the final lags
  even though it fits within `n // 3`.
- **`heavy_nonseasonal_lag`**: the largest **non-seasonal** lag consumes more than
  15% of the series. Seasonal lags (period or low harmonic, e.g. weekly lag 52)
  are *expected* to be large on short series, so they are excluded — only an
  isolated long PACF lag with no seasonal explanation is flagged (it drops that
  many training rows via `window_size = max(lags)`).
- **`too_many_lags`**: the lag set fills more than half of the allowed budget
  (`n_lags / (n // 3) > 0.5`) **and** there are more than 15 lags. This relative
  density replaces a flat count so naturally lag-rich fine frequencies (minutely,
  hourly) are not punished for having many significant lags.
- **`empty`**: no lags selected at all.
- **`lag_exceeds_constraint`**: `max_lag > n // 3` (should never happen).

Extra context columns: `max_nonseasonal_lag` (the offending lag, if any) and
`lag_density` (`n_lags / (n // 3)`).


In [11]:
def seasonal_lag_set(
    seasonalities: list[int],
    max_lag_allowed: int,
    harmonics: tuple[int, ...] = (1, 2, 3),
) -> set[int]:
    """
    Lags explained by seasonality: each period and its low harmonics that
    still fit within `max_lag_allowed`.
    """
    out: set[int] = set()
    for period in seasonalities or []:
        for k in harmonics:
            lag = k * period
            if lag <= max_lag_allowed:
                out.add(lag)
    return out


def diagnose(row: dict, n_key: str) -> dict:
    n = row[n_key]
    lags = row["final_lags"]
    seas = row["seasonalities"]
    primary = seas[0] if seas else None
    max_lag = max(lags) if lags else None
    max_lag_allowed = max(n // 3, 1)

    # A lag is "seasonal" if it is within +/-1 of a seasonal period or one of
    # its low harmonics. Such lags are expected to be large on short series.
    seasonal = seasonal_lag_set(seas, max_lag_allowed)
    nonseasonal = [
        lag for lag in lags if not any(abs(lag - s) <= 1 for s in seasonal)
    ]
    max_nonseasonal = max(nonseasonal) if nonseasonal else None

    missing_primary = bool(
        primary is not None and primary <= max_lag_allowed and primary not in lags
    )
    # Only a NON-seasonal long lag is suspicious: a seasonal lag (e.g. weekly
    # 52) is deliberately enriched and is large by design on short series.
    heavy_nonseasonal_lag = bool(
        max_nonseasonal is not None and max_nonseasonal > 0.15 * n
    )
    # Relative density instead of a flat count: many lags is only suspicious
    # when they fill a large share of the allowed budget. Fine frequencies
    # (minutely, hourly) are naturally lag-rich and should not be punished.
    lag_density = len(lags) / max_lag_allowed
    too_many = bool(lag_density > 0.5 and len(lags) > 15)
    empty = len(lags) == 0
    exceeds = bool(max_lag is not None and max_lag > max_lag_allowed)

    return {
        "case": row["case"],
        "task": row["task"],
        "n": n,
        "primary_season": primary,
        "max_lag": max_lag,
        "max_nonseasonal_lag": max_nonseasonal,
        "n_lags": len(lags),
        "lag_density": round(lag_density, 2),
        "missing_primary": missing_primary,
        "heavy_nonseasonal_lag": heavy_nonseasonal_lag,
        "too_many_lags": too_many,
        "empty": empty,
        "lag_exceeds_constraint": exceeds,
    }


diag_rows = (
    [diagnose(r, "n") for r in rows]
    + [diagnose(r, "n") for r in multi_rows]
    + [diagnose(r, "n") for r in mv_rows]
    + [diagnose(r, "n_shortest") for r in long_rows]
)
diag_df = pd.DataFrame(diag_rows)

flag_cols = ["missing_primary", "heavy_nonseasonal_lag", "too_many_lags", "empty", "lag_exceeds_constraint"]
diag_df["any_flag"] = diag_df[flag_cols].any(axis=1)
diag_df


,case,task,n,primary_season,max_lag,max_nonseasonal_lag,n_lags,lag_density,missing_primary,heavy_nonseasonal_lag,too_many_lags,empty,lag_exceeds_constraint,any_flag
0,minutely_3w,single_series,7560,60,60,58,47,0.02,False,False,False,False,False,False
1,hourly_3mo,single_series,2160,24,30,30,17,0.02,False,False,False,False,False,False
2,daily_4y,single_series,1460,7,20,12,6,0.01,False,False,False,False,False,False
3,daily_2y,single_series,730,7,20,1,4,0.02,False,False,False,False,False,False
4,weekly_5y,single_series,260,52,52,5,6,0.07,False,False,False,False,False,False
5,monthly_10y,single_series,120,12,12,4,5,0.12,False,False,False,False,False,False
6,quarterly_15y,single_series,60,4,5,2,4,0.20,False,False,False,False,False,False
7,yearly_50y,single_series,50,1,5,5,5,0.31,False,False,False,False,False,False
8,hourly_multi,multi_series,1440,24,24,19,12,0.03,False,False,False,False,False,False
9,daily_multi,multi_series,1095,7,20,19,8,0.02,False,False,False,False,False,False


In [12]:
# Only the flagged cases — these are the ones to investigate.
flagged = diag_df[diag_df["any_flag"]]
if flagged.empty:
    print("No strange lag selections flagged across all cases.")
else:
    print(f"{len(flagged)} flagged case(s):")
flagged

No strange lag selections flagged across all cases.


,case,task,n,primary_season,max_lag,max_nonseasonal_lag,n_lags,lag_density,missing_primary,heavy_nonseasonal_lag,too_many_lags,empty,lag_exceeds_constraint,any_flag


## 7. Notes

- `final_lags` columns above are printed in full so you can eyeball the actual
  selections, not just the summary counts.
- To stress a specific scenario, tweak the `*_cases` lists (length, frequency,
  number of series) and re-run from that section.
- The diagnostics are heuristics, not assertions — a flag means "look closer",
  not "definitely wrong".